# Chapter 1: PyTorch and Object-Oriented Programming
**Module 02: Intermediate Deep Learning with PyTorch**  
*Instructor: Michal Oleszak, Machine Learning Engineer*

> Premium course notebook integrating the full PDF flow: OOP, custom datasets, class-based models, training, optimizers, evaluation, and techniques for vanishing/exploding gradients.


## Learning Objectives
- Use OOP to define objects with attributes and methods.
- Implement a custom PyTorch `Dataset` and wrap it in a `DataLoader`.
- Compare `nn.Sequential` and class-based `nn.Module` models.
- Train and evaluate a binary classifier.
- Compare SGD, Adagrad, RMSprop, and Adam.
- Apply initialization, activation, and batch-normalization strategies for unstable gradients.

## Prerequisites
| Area | Expected Knowledge |
|---|---|
| Neural-network training | Forward pass, loss calculation, backward pass |
| PyTorch workflow | Datasets, DataLoaders, training loop, evaluation loop |


## 1. Object-Oriented Programming (OOP)
PyTorch uses OOP heavily because datasets and models combine **data** with **behavior**.

| OOP Concept | Meaning | PyTorch Parallel |
|---|---|---|
| Object | Entity with state and behavior | Dataset or model instance |
| Attribute | Data stored on the object | `self.balance`, `self.fc1` |
| Method | Function attached to the object | `deposit()`, `forward()` |
| `__init__()` | Constructor called when object is created | Define layers or load data |


In [ ]:
# PDF example: attribute initialized in __init__
class BankAccount:
    def __init__(self, balance):
        self.balance = balance

account = BankAccount(100)
print(account.balance)


In [ ]:
# PDF example: method that changes object state
class BankAccount:
    def __init__(self, balance):
        self.balance = balance

    def deposit(self, amount):
        self.balance += amount

account = BankAccount(100)
account.deposit(50)
print(account.balance)


## 2. Custom PyTorch Dataset
The water-potability dataset contains water-quality measurements and a binary target.

| Method | Role |
|---|---|
| `__init__()` | Load data and store it as an array/tensor |
| `__len__()` | Return number of samples |
| `__getitem__(idx)` | Return one feature-label pair |

> `super().__init__()` keeps the custom dataset aligned with PyTorch's `Dataset` behavior.


In [ ]:
from pathlib import Path
import zipfile
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

DATA_DIR = Path("datasets")
WATER_ZIP = DATA_DIR / "water_potability.zip"
WATER_DIR = DATA_DIR / "water_potability"
if WATER_ZIP.exists() and not WATER_DIR.exists():
    with zipfile.ZipFile(WATER_ZIP) as zf:
        zf.extractall(DATA_DIR)

class WaterDataset(Dataset):
    def __init__(self, csv_path):
        super().__init__()
        df = pd.read_csv(csv_path)
        df = df.fillna(df.mean(numeric_only=True))
        self.data = df.to_numpy(dtype="float32")

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, idx):
        features = self.data[idx, :-1]
        label = self.data[idx, -1]
        return torch.tensor(features, dtype=torch.float32), torch.tensor(label, dtype=torch.float32)

if (WATER_DIR / "water_train.csv").exists():
    dataset_train = WaterDataset(WATER_DIR / "water_train.csv")
    dataloader_train = DataLoader(dataset_train, batch_size=2, shuffle=True)
    features, labels = next(iter(dataloader_train))
    print(f"Features: {features}\nLabels: {labels}")
else:
    print("Water CSV not found; class is ready for any compatible CSV.")


## 3. PyTorch Model: Sequential vs. Class-Based
| Style | Best Use | Limitation |
|---|---|---|
| `nn.Sequential` | Straight-line layer stacks | Less flexible for custom logic |
| `nn.Module` class | Reusable layers and custom forward pass | Slightly more verbose |


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

sequential_net = nn.Sequential(
    nn.Linear(9, 16), nn.ReLU(),
    nn.Linear(16, 8), nn.ReLU(),
    nn.Linear(8, 1), nn.Sigmoid(),
)
print(sequential_net)


In [ ]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(9, 16)
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = torch.sigmoid(self.fc3(x))
        return x

net = Net()
print(net)


## 4. Training Loop
Binary classification uses `nn.BCELoss()` with a sigmoid output.

| Step | Operation |
|---|---|
| Clear gradients | `optimizer.zero_grad()` |
| Forward pass | `outputs = net(features)` |
| Compute loss | `criterion(outputs, labels.view(-1, 1))` |
| Backpropagate | `loss.backward()` |
| Update weights | `optimizer.step()` |


In [ ]:
import torch.optim as optim
criterion = nn.BCELoss()
optimizer = optim.SGD(net.parameters(), lr=0.01)

if 'dataloader_train' in globals():
    net.train()
    for epoch in range(2):
        total_loss = 0.0
        for features, labels in dataloader_train:
            optimizer.zero_grad()
            outputs = net(features)
            loss = criterion(outputs, labels.view(-1, 1))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}/2 | loss={total_loss/len(dataloader_train):.4f}")
else:
    print("Create dataloader_train to run training.")


## 5. Optimizers
| Optimizer | PDF Takeaway |
|---|---|
| SGD | Simple and efficient for basic models; rarely used as the default in practice |
| Adagrad | Per-parameter learning rates; useful for sparse data; can shrink too fast |
| RMSprop | Adapts updates using previous gradient sizes |
| Adam | RMSprop plus momentum; versatile and widely used |


In [ ]:
optimizer_sgd = optim.SGD(net.parameters(), lr=0.01)
optimizer_adagrad = optim.Adagrad(net.parameters(), lr=0.01)
optimizer_rmsprop = optim.RMSprop(net.parameters(), lr=0.01)
optimizer_adam = optim.Adam(net.parameters(), lr=0.01)
print([type(o).__name__ for o in [optimizer_sgd, optimizer_adagrad, optimizer_rmsprop, optimizer_adam]])


## 6. Model Evaluation
Evaluation should use `net.eval()` and `torch.no_grad()`. For binary classification, convert probabilities to labels with a threshold such as `0.5`.


In [ ]:
try:
    from torchmetrics import Accuracy
except ImportError:
    Accuracy = None

if (WATER_DIR / "water_test.csv").exists() and Accuracy is not None:
    dataset_test = WaterDataset(WATER_DIR / "water_test.csv")
    dataloader_test = DataLoader(dataset_test, batch_size=64, shuffle=False)
    acc = Accuracy(task="binary")
    net.eval()
    with torch.no_grad():
        for features, labels in dataloader_test:
            outputs = net(features)
            preds = (outputs >= 0.5).float()
            acc(preds, labels.view(-1, 1))
    print(f"Accuracy: {acc.compute()}")
else:
    print("Install torchmetrics and provide water_test.csv to run the PDF evaluation loop.")


## 7. Vanishing and Exploding Gradients
| Problem | What Happens | Symptom |
|---|---|---|
| Vanishing gradients | Gradients shrink during backpropagation | Earlier layers learn slowly or not at all |
| Exploding gradients | Gradients grow too large | Loss diverges or updates become unstable |

PDF remedies: **proper weight initialization**, **good activations**, and **batch normalization**.


### 7.1 Weight Initialization
Good initialization keeps activation and gradient variance stable across layers. For ReLU-like activations, use He/Kaiming initialization.


In [ ]:
import torch.nn.init as init
layer = nn.Linear(8, 1)
print("Before:", layer.weight)
init.kaiming_uniform_(layer.weight)
print("After:", layer.weight)


In [ ]:
class KaimingNet(nn.Module):
    def __init__(self):
        super(KaimingNet, self).__init__()
        self.fc1 = nn.Linear(9, 16)
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 1)
        init.kaiming_uniform_(self.fc1.weight)
        init.kaiming_uniform_(self.fc2.weight)
        init.kaiming_uniform_(self.fc3.weight, nonlinearity="sigmoid")

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return torch.sigmoid(self.fc3(x))

print(KaimingNet())


### 7.2 Activation Functions
| Activation | PDF Note |
|---|---|
| `relu()` | Common default, but zero for negative inputs can cause dying neurons |
| `elu()` | Non-zero negative-side gradients and average output closer to zero |


In [ ]:
x = torch.linspace(-3, 3, steps=7)
print("x   ", x)
print("relu", F.relu(x))
print("elu ", F.elu(x))


### 7.3 Batch Normalization
Batch normalization normalizes a layer's output, then learns scale and shift parameters. It can speed up loss decrease and help against unstable gradients.


In [ ]:
class BatchNormNet(nn.Module):
    def __init__(self):
        super(BatchNormNet, self).__init__()
        self.fc1 = nn.Linear(9, 16)
        self.bn1 = nn.BatchNorm1d(16)
        self.fc2 = nn.Linear(16, 8)
        self.bn2 = nn.BatchNorm1d(8)
        self.fc3 = nn.Linear(8, 1)

    def forward(self, x):
        x = F.elu(self.bn1(self.fc1(x)))
        x = F.elu(self.bn2(self.fc2(x)))
        return torch.sigmoid(self.fc3(x))

print(BatchNormNet())


## Chapter Summary
- OOP lets PyTorch datasets and models package data with behavior.
- Custom datasets implement `__init__`, `__len__`, and `__getitem__`.
- Training loops follow the sequence: zero gradients, forward pass, loss, backward pass, optimizer step.
- Adam is a common default, but optimizer behavior matters.
- Initialization, activation choice, and batch normalization help stabilize gradients.
